In [ ]:
import pandas as pd
import glob
import os

folder_path = "Missing-Data"   # folder name

all_files = glob.glob(os.path.join(folder_path, "*.xls"))

dataframes = {}   # dict to store all dataframes

for file in all_files:
    name = os.path.splitext(os.path.basename(file))[0]   # file name without extension
    dataframes[name] = pd.read_excel(file)

# example usage:
# df1 = dataframes['filename']
# list all dataframes names:
print(dataframes.keys())


In [ ]:
# you already have: dataframes = { 'Adolescents_out_school': df, ... }

# groups by DICT KEYS (not by values inside the frames)
group_2000_2023 = [
    "Adolescents_out_school",
    "Children_out_school_primary",
    "Gov_expenditure",
    "Lower_secondary_completion",
    "Lower_secondary_completion"  # duplicate in your list; harmless
]

group_2000_2018 = [
    "Pupil_teacher_primary",
    "Pupil-teacher_secondary",
    "School_enrollment_primary",
    "School_enrollment_secondary"
]

base_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

def safe_subset(df, base, start_year, end_year):
    years = [str(y) for y in range(start_year, end_year + 1)]
    keep = [c for c in base + years if c in df.columns]
    return df.loc[:, keep]

filtered_dataframes = {}

for key, df in dataframes.items():
    if key in group_2000_2023:
        filtered = safe_subset(df, base_cols, 2001, 2023)
        filtered_dataframes[key] = filtered
        dataframes[key] = filtered  # overwrite original if you want
    elif key in group_2000_2018:
        filtered = safe_subset(df, base_cols, 2000, 2018)
        filtered_dataframes[key] = filtered
        dataframes[key] = filtered  # overwrite original if you want
    else:
        # keys like 'Primary_completion' are ignored per your spec
        continue

print("Filtered frames created:", list(filtered_dataframes.keys()))


In [ ]:
for key, df in filtered_dataframes.items():
    filtered_dataframes[key] = df.dropna(how='any')

In [ ]:
for key, df in filtered_dataframes.items():
     print(filtered_dataframes[key].shape)
    

In [ ]:
import matplotlib.pyplot as plt

group1 = [
    "Adolescents_out_school",
    "Children_out_school_primary",
    "Gov_expenditure",
    "Lower_secondary_completion"
]

years = [str(y) for y in range(2001,2024)]

for key in group1:
    df = filtered_dataframes[key]
    
    plt.figure(figsize=(12,6))
    
    for idx,row in df.iterrows():
        plt.plot(years, row[years].values, label=row['Country Name'])
    
    plt.xlabel("Year")
    plt.ylabel("Value")
    plt.title(df['Indicator Name'].iloc[0])   # same indicator for all rows inside same df
    plt.legend(loc='best', fontsize=6)
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

group2 = [
    "Pupil_teacher_primary",
    "Pupil-teacher_secondary",
    "School_enrollment_primary",
    "School_enrollment_secondary"
]

years2 = [str(y) for y in range(2000,2019)]

for key in group2:
    df = filtered_dataframes[key]
    
    plt.figure(figsize=(12,6))
    
    for idx,row in df.iterrows():
        plt.plot(years2, row[years2].values, label=row['Country Name'])
    
    plt.xlabel("Year")
    plt.ylabel("Value")
    plt.title(df['Indicator Name'].iloc[0])   # indicator name for this entire df
    plt.legend(loc='best', fontsize=6)
    plt.tight_layout()
    plt.show()


In [ ]:
filtered_dataframes['Adolescents_out_school'].shape

In [ ]:
import numpy as np

Ado_out_school = filtered_dataframes['Adolescents_out_school'].copy()

# drop useless identity decorations
Ado_out_school = Ado_out_school.drop(columns=['Country Name','Indicator Name','Indicator Code'])

# set index to Country Code
Ado_out_school = Ado_out_school.set_index('Country Code')
Ado_out_school_Missing = Ado_out_school.copy()


In [ ]:
Ado_out_school = Ado_out_school.rename(
    columns = lambda c: f"y{c}" if str(c).isdigit() else c
)

Ado_out_school_Missing = Ado_out_school_Missing.rename(
    columns = lambda c: f"y{c}" if str(c).isdigit() else c
)


Ado_out_school.head()

In [ ]:
print(Ado_out_school.columns)
print(Ado_out_school.dtypes)

In [ ]:
Ado_out_school_Missing.head()

In [ ]:
# create mask of same shape as dataframe
nan_indices = np.random.binomial(n=1, p=0.2, size=Ado_out_school.size).reshape(Ado_out_school.shape).astype(bool)
Ado_out_school_Missing[nan_indices]= np.nan

In [ ]:
Ado_out_school_Missing.head()

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, Tuple, List, Optional
from pathlib import Path

# Multiple Imputation via IterativeImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

# -----------------------------
# Helpers
# -----------------------------
def _ensure_yyear_columns(df: pd.DataFrame, year_start: int, year_end: int) -> pd.Index:
    """
    Return the y-prefixed year columns present in df: ['y2001', ..., 'y2023'].
    If plain '2001' exists, accept and convert to 'y2001'. Always return strings.
    """
    found: List[str] = []
    for c in df.columns:
        sc = str(c)
        if sc.startswith("y") and sc[1:].isdigit():
            yr = int(sc[1:])
            if year_start <= yr <= year_end:
                found.append(f"y{yr}")
        elif sc.isdigit():
            yr = int(sc)
            if year_start <= yr <= year_end:
                found.append(f"y{yr}")
    found = sorted(set(found), key=lambda s: int(s[1:]))
    return pd.Index(found, dtype=object)

def _metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    diff = y_pred - y_true
    mae = float(np.mean(np.abs(diff)))
    rmse = float(np.sqrt(np.mean(diff**2)))

    mask_nonzero = (np.abs(y_true) > 1e-12)
    if mask_nonzero.any():
        mape = float(np.mean(np.abs(diff[mask_nonzero] / y_true[mask_nonzero])) * 100.0)
    else:
        mape = float('nan')

    denom = (np.abs(y_true) + np.abs(y_pred))
    denom_safe = np.where(denom == 0, 1.0, denom)
    smape = float(np.mean(2.0 * np.abs(diff) / denom_safe) * 100.0)

    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "SMAPE_%": smape}

def _maybe_save(df: pd.DataFrame, path: Optional[Path]) -> None:
    if path is None:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path)

# -----------------------------
# Imputation methods
# -----------------------------
def mean_impute(df: pd.DataFrame, year_cols: pd.Index) -> pd.DataFrame:
    out = df.copy()
    means = out[year_cols].mean(skipna=True)
    out[year_cols] = out[year_cols].fillna(means)
    return out

def median_impute(df: pd.DataFrame, year_cols: pd.Index) -> pd.DataFrame:
    out = df.copy()
    meds = out[year_cols].median(skipna=True)
    out[year_cols] = out[year_cols].fillna(meds)
    return out

def mode_impute(df: pd.DataFrame, year_cols: pd.Index) -> pd.DataFrame:
    out = df.copy()
    modes = {}
    for c in year_cols:
        s = out[c].dropna()
        modes[c] = s.mode(dropna=True).iloc[0] if len(s) > 0 else np.nan
    out[year_cols] = out[year_cols].fillna(pd.Series(modes))
    return out

def hot_deck_impute(df: pd.DataFrame, year_cols: pd.Index, random_state: int = 0) -> pd.DataFrame:
    """Random hot-deck per column: sample donors from observed values with replacement."""
    rng = np.random.default_rng(random_state)
    out = df.copy()
    for c in year_cols:
        s = out[c]
        miss = s.isna()
        donors = s.dropna().to_numpy()
        if donors.size > 0 and miss.any():
            out.loc[miss, c] = rng.choice(donors, size=int(miss.sum()), replace=True)
    return out

def linear_interp_impute(df: pd.DataFrame, year_cols: pd.Index) -> pd.DataFrame:
    """Linear interpolation across time along each row."""
    out = df.copy()
    sorted_cols = sorted(list(year_cols), key=lambda s: int(s[1:]))
    tmp = out[sorted_cols].astype(float)
    tmp = tmp.interpolate(method="linear", axis=1, limit_direction="both")
    out[sorted_cols] = tmp
    return out

def multiple_impute_iterative(
    df: pd.DataFrame,
    year_cols: pd.Index,
    n_imputations: int = 5,
    random_state: int = 0
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    Multiple imputation via IterativeImputer with posterior sampling.
    Returns averaged imputed df plus individual draws in a dict.
    """
    X = df[year_cols].to_numpy(dtype=float)
    imputed_arrays = []
    imputations: Dict[str, pd.DataFrame] = {}

    for k in range(n_imputations):
        imp = IterativeImputer(
            random_state=random_state + k,
            sample_posterior=True,
            max_iter=20,
            initial_strategy="mean",
            skip_complete=True,
        )
        X_imp = imp.fit_transform(X)
        imputed_arrays.append(X_imp)

        tmp_df = df.copy()
        tmp_df[year_cols] = X_imp
        imputations[f"MI_draw_{k+1}"] = tmp_df

    X_avg = np.mean(imputed_arrays, axis=0)
    out = df.copy()
    out[year_cols] = X_avg
    return out, imputations

# -----------------------------
# Orchestrator + Evaluation
# -----------------------------
def run_imputations_and_evaluate(
    Ado_out_school_Missing: pd.DataFrame,
    Ado_out_school: pd.DataFrame,
    year_start: int = 2001,
    year_end: int = 2023,
    random_state: int = 0,
    n_imputations: int = 5,
    save_dir: Optional[str] = None
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    Assumes year columns are y-prefixed (y2001..y2023).
    Compares imputations only at positions that were NaN in Ado_out_school_Missing.
    Prints metrics and returns (metrics_df, imputed_results).
    If save_dir is provided, CSVs are saved there; directory is created if missing.
    """

    # Detect year columns (y-prefixed) from both frames and intersect
    yc_miss = _ensure_yyear_columns(Ado_out_school_Missing, year_start, year_end)
    yc_true = _ensure_yyear_columns(Ado_out_school, year_start, year_end)
    year_cols = yc_miss.intersection(yc_true)
    if len(year_cols) == 0:
        raise ValueError("No overlapping y-year columns found between the two dataframes.")

    # Align index
    common_index = Ado_out_school_Missing.index.intersection(Ado_out_school.index)
    miss_df = Ado_out_school_Missing.loc[common_index, year_cols].astype(float)
    true_df = Ado_out_school.loc[common_index, year_cols].astype(float)

    # Only evaluate where the synthetic mask produced NaNs
    eval_mask = miss_df.isna().to_numpy()

    imputed_results: Dict[str, pd.DataFrame] = {}
    rows = []

    # Mean
    mean_df = mean_impute(miss_df, year_cols)
    y_true = true_df.to_numpy()[eval_mask]
    y_pred = mean_df.to_numpy()[eval_mask]
    rows.append({"Method": "Mean", **_metrics(y_true, y_pred)})
    imputed_results["Mean"] = mean_df

    # Median
    median_df = median_impute(miss_df, year_cols)
    y_pred = median_df.to_numpy()[eval_mask]
    rows.append({"Method": "Median", **_metrics(y_true, y_pred)})
    imputed_results["Median"] = median_df

    # Mode
    mode_df = mode_impute(miss_df, year_cols)
    y_pred = mode_df.to_numpy()[eval_mask]
    rows.append({"Method": "Mode", **_metrics(y_true, y_pred)})
    imputed_results["Mode"] = mode_df

    # Hot-Deck
    hotdeck_df = hot_deck_impute(miss_df, year_cols, random_state=random_state)
    y_pred = hotdeck_df.to_numpy()[eval_mask]
    rows.append({"Method": "Hot-Deck", **_metrics(y_true, y_pred)})
    imputed_results["HotDeck"] = hotdeck_df

    # Linear Interpolation
    lin_df = linear_interp_impute(miss_df, year_cols)
    y_pred = lin_df.to_numpy()[eval_mask]
    rows.append({"Method": "Linear Interpolation", **_metrics(y_true, y_pred)})
    imputed_results["LinearInterpolation"] = lin_df

    # Multiple Imputation
    mi_avg_df, mi_draws = multiple_impute_iterative(
        miss_df, year_cols, n_imputations=n_imputations, random_state=random_state
    )
    y_pred = mi_avg_df.to_numpy()[eval_mask]
    rows.append({"Method": f"Multiple Imputation (avg {n_imputations})", **_metrics(y_true, y_pred)})
    imputed_results["MultipleImputation_avg"] = mi_avg_df
    imputed_results.update(mi_draws)

    metrics_df = pd.DataFrame(rows).set_index("Method")

    # Save if asked, otherwise skip silently
    save_base = Path(save_dir) if save_dir is not None else None
    _maybe_save(metrics_df, save_base / "ado_imputation_metrics_ycols.csv" if save_base else None)
    _maybe_save(imputed_results["Mean"], save_base / "ado_imputed_mean.csv" if save_base else None)
    _maybe_save(imputed_results["Median"], save_base / "ado_imputed_median.csv" if save_base else None)
    _maybe_save(imputed_results["Mode"], save_base / "ado_imputed_mode.csv" if save_base else None)
    _maybe_save(imputed_results["HotDeck"], save_base / "ado_imputed_hotdeck.csv" if save_base else None)
    _maybe_save(imputed_results["LinearInterpolation"], save_base / "ado_imputed_linear.csv" if save_base else None)
    _maybe_save(imputed_results["MultipleImputation_avg"], save_base / "ado_imputed_MI_avg.csv" if save_base else None)

    print("\n=== Ado_out_school Imputation Metrics (y2001..y2023) ===\n")
    print(metrics_df)

    return metrics_df, imputed_results


In [ ]:
metrics, results = run_imputations_and_evaluate(
    Ado_out_school_Missing, Ado_out_school,
    year_start=2001, year_end=2023,
    random_state=42, n_imputations=5,
    save_dir="."
)

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, Tuple, List, Optional
from pathlib import Path

# Multiple Imputation via IterativeImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer

# -----------------------------
# Helpers
# -----------------------------
def _ensure_yyear_columns(df: pd.DataFrame, year_start: int, year_end: int) -> pd.Index:
    """
    Return the y-prefixed year columns present in df: ['y2001', ..., 'y2023'].
    If plain '2001' exists, accept and convert to 'y2001'. Always return strings.
    """
    found: List[str] = []
    for c in df.columns:
        sc = str(c)
        if sc.startswith("y") and sc[1:].isdigit():
            yr = int(sc[1:])
            if year_start <= yr <= year_end:
                found.append(f"y{yr}")
        elif sc.isdigit():
            yr = int(sc)
            if year_start <= yr <= year_end:
                found.append(f"y{yr}")
    found = sorted(set(found), key=lambda s: int(s[1:]))
    return pd.Index(found, dtype=object)

def _metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    diff = y_pred - y_true
    mae = float(np.mean(np.abs(diff)))
    rmse = float(np.sqrt(np.mean(diff**2)))

    mask_nonzero = (np.abs(y_true) > 1e-12)
    if mask_nonzero.any():
        mape = float(np.mean(np.abs(diff[mask_nonzero] / y_true[mask_nonzero])) * 100.0)
    else:
        mape = float('nan')

    denom = (np.abs(y_true) + np.abs(y_pred))
    denom_safe = np.where(denom == 0, 1.0, denom)
    smape = float(np.mean(2.0 * np.abs(diff) / denom_safe) * 100.0)

    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "SMAPE_%": smape}

def _maybe_save(df: pd.DataFrame, path: Optional[Path]) -> None:
    if path is None:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path)

# -----------------------------
# Imputation methods (simple)
# -----------------------------
def mean_impute(df: pd.DataFrame, year_cols: pd.Index) -> pd.DataFrame:
    out = df.copy()
    means = out[year_cols].mean(skipna=True)
    out[year_cols] = out[year_cols].fillna(means)
    return out

def median_impute(df: pd.DataFrame, year_cols: pd.Index) -> pd.DataFrame:
    out = df.copy()
    meds = out[year_cols].median(skipna=True)
    out[year_cols] = out[year_cols].fillna(meds)
    return out

def mode_impute(df: pd.DataFrame, year_cols: pd.Index) -> pd.DataFrame:
    out = df.copy()
    modes = {}
    for c in year_cols:
        s = out[c].dropna()
        modes[c] = s.mode(dropna=True).iloc[0] if len(s) > 0 else np.nan
    out[year_cols] = out[year_cols].fillna(pd.Series(modes))
    return out

def hot_deck_impute(df: pd.DataFrame, year_cols: pd.Index, random_state: int = 0) -> pd.DataFrame:
    """Random hot-deck per column: sample donors from observed values with replacement."""
    rng = np.random.default_rng(random_state)
    out = df.copy()
    for c in year_cols:
        s = out[c]
        miss = s.isna()
        donors = s.dropna().to_numpy()
        if donors.size > 0 and miss.any():
            out.loc[miss, c] = rng.choice(donors, size=int(miss.sum()), replace=True)
    return out

def linear_interp_impute(df: pd.DataFrame, year_cols: pd.Index) -> pd.DataFrame:
    """Linear interpolation across time along each row."""
    out = df.copy()
    sorted_cols = sorted(list(year_cols), key=lambda s: int(s[1:]))
    tmp = out[sorted_cols].astype(float)
    tmp = tmp.interpolate(method="linear", axis=1, limit_direction="both")
    out[sorted_cols] = tmp
    return out

def multiple_impute_iterative(
    df: pd.DataFrame,
    year_cols: pd.Index,
    n_imputations: int = 5,
    random_state: int = 0
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    Multiple imputation via IterativeImputer with posterior sampling.
    Returns averaged imputed df plus individual draws in a dict.
    """
    X = df[year_cols].to_numpy(dtype=float)
    imputed_arrays = []
    imputations: Dict[str, pd.DataFrame] = {}

    for k in range(n_imputations):
        imp = IterativeImputer(
            random_state=random_state + k,
            sample_posterior=True,
            max_iter=20,
            initial_strategy="mean",
            skip_complete=True,
        )
        X_imp = imp.fit_transform(X)
        imputed_arrays.append(X_imp)

        tmp_df = df.copy()
        tmp_df[year_cols] = X_imp
        imputations[f"MI_draw_{k+1}"] = tmp_df

    X_avg = np.mean(imputed_arrays, axis=0)
    out = df.copy()
    out[year_cols] = X_avg
    return out, imputations

# -----------------------------
# KNN Imputation
# -----------------------------
def knn_impute(
    df: pd.DataFrame,
    year_cols: pd.Index,
    n_neighbors: int = 5,
    weights: str = "uniform"
) -> pd.DataFrame:
    """
    KNNImputer across columns: each column value is imputed by neighbors in feature space
    (rows are samples, columns are years).
    """
    out = df.copy()
    imputer = KNNImputer(n_neighbors=n_neighbors, weights=weights)
    arr = imputer.fit_transform(out[year_cols].to_numpy(dtype=float))
    out[year_cols] = arr
    return out

# -----------------------------
# LSTM Imputation (sequence autoencoder)
# -----------------------------
def lstm_impute(
    df: pd.DataFrame,
    year_cols: pd.Index,
    random_state: int = 0,
    epochs: int = 200,
    batch_size: int = 64,
    hidden_units: int = 32,
    verbose: int = 0
) -> pd.DataFrame:
    """
    Train a small LSTM sequence autoencoder on all rows (time = years, features=1).
    Loss is computed only on observed entries (mask); predictions fill NaNs.
    If TensorFlow is missing, raises a helpful error.
    """
    try:
        import tensorflow as tf
        from tensorflow import keras
        from tensorflow.keras import layers
    except Exception as e:
        raise ImportError(
            "TensorFlow/Keras not available. Install tensorflow to use lstm_impute()."
        ) from e

    # Reproducibility-ish
    np.random.seed(random_state)
    tf.random.set_seed(random_state)

    out = df.copy()
    cols_sorted = sorted(list(year_cols), key=lambda s: int(s[1:]))
    X = out[cols_sorted].to_numpy(dtype=float)

    # Build masks: 1 for observed, 0 for missing
    obs_mask = (~np.isnan(X)).astype(float)

    # Simple normalization per column to help training
    col_means = np.nanmean(X, axis=0)
    col_stds = np.nanstd(X, axis=0)
    col_stds = np.where(col_stds < 1e-8, 1.0, col_stds)
    X_norm = (X - col_means) / col_stds
    X_norm = np.nan_to_num(X_norm, nan=0.0)

    # Shape to [n_samples, timesteps, features]
    n_samples, T = X_norm.shape
    X_seq = X_norm.reshape(n_samples, T, 1)
    M_seq = obs_mask.reshape(n_samples, T, 1)

    # Model: encoder-decoder with masking via sample_weight on observed entries
    inp = layers.Input(shape=(T, 1))
    # A small stack
    x = layers.LSTM(hidden_units, return_sequences=True)(inp)
    x = layers.LSTM(hidden_units, return_sequences=True)(x)
    out_seq = layers.TimeDistributed(layers.Dense(1))(x)

    model = keras.Model(inp, out_seq)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

    # sample_weight: 3D not allowed; flatten time dimension with custom loss via masking callback
    # Instead, we weight per time-step by providing a 2D weight matrix through a custom generator.
    # We'll train with a small custom data generator that yields (X, y, weights) where y = X and
    # weights = obs_mask (squeezed).

    y_seq = X_seq.copy()  # reconstruct observed sequence

    def data_gen(Xs, Ys, Ms, bs):
        n = Xs.shape[0]
        idx = np.arange(n)
        while True:
            np.random.shuffle(idx)
            for i in range(0, n, bs):
                j = idx[i:i+bs]
                yield Xs[j], Ys[j], Ms[j].squeeze(-1)

    steps = int(np.ceil(n_samples / batch_size))
    model.fit(
        data_gen(X_seq, y_seq, M_seq, batch_size),
        steps_per_epoch=steps,
        epochs=epochs,
        verbose=verbose
    )

    # Predict full sequences
    X_pred = model.predict(X_seq, verbose=0).reshape(n_samples, T)

    # De-normalize
    X_pred_denorm = X_pred * col_stds + col_means

    # Fill only missing spots with predictions
    X_filled = np.where(np.isnan(X), X_pred_denorm, X)

    out[cols_sorted] = X_filled
    return out

# -----------------------------
# Orchestrator + Evaluation
# -----------------------------
def run_imputations_and_evaluate(
    Ado_out_school_Missing: pd.DataFrame,
    Ado_out_school: pd.DataFrame,
    year_start: int = 2001,
    year_end: int = 2023,
    random_state: int = 0,
    n_imputations: int = 5,
    save_dir: Optional[str] = None,
    knn_neighbors: int = 5,
    lstm_epochs: int = 200,
    lstm_hidden: int = 32,
    lstm_batch: int = 64,
    lstm_verbose: int = 0
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    Assumes year columns are y-prefixed (y2001..y2023).
    Compares imputations only at positions that were NaN in Ado_out_school_Missing.
    Prints metrics and returns (metrics_df, imputed_results).
    If save_dir is provided, CSVs are saved there; directory is created if missing.
    """

    # Detect year columns (y-prefixed) from both frames and intersect
    yc_miss = _ensure_yyear_columns(Ado_out_school_Missing, year_start, year_end)
    yc_true = _ensure_yyear_columns(Ado_out_school, year_start, year_end)
    year_cols = yc_miss.intersection(yc_true)
    if len(year_cols) == 0:
        raise ValueError("No overlapping y-year columns found between the two dataframes.")

    # Align index
    common_index = Ado_out_school_Missing.index.intersection(Ado_out_school.index)
    miss_df = Ado_out_school_Missing.loc[common_index, year_cols].astype(float)
    true_df = Ado_out_school.loc[common_index, year_cols].astype(float)

    # Only evaluate where the synthetic mask produced NaNs
    eval_mask = miss_df.isna().to_numpy()

    imputed_results: Dict[str, pd.DataFrame] = {}
    rows = []

    # Mean
    mean_df = mean_impute(miss_df, year_cols)
    y_true = true_df.to_numpy()[eval_mask]
    y_pred = mean_df.to_numpy()[eval_mask]
    rows.append({"Method": "Mean", **_metrics(y_true, y_pred)})
    imputed_results["Mean"] = mean_df

    # Median
    median_df = median_impute(miss_df, year_cols)
    y_pred = median_df.to_numpy()[eval_mask]
    rows.append({"Method": "Median", **_metrics(y_true, y_pred)})
    imputed_results["Median"] = median_df

    # Mode
    mode_df = mode_impute(miss_df, year_cols)
    y_pred = mode_df.to_numpy()[eval_mask]
    rows.append({"Method": "Mode", **_metrics(y_true, y_pred)})
    imputed_results["Mode"] = mode_df

    # Hot-Deck
    hotdeck_df = hot_deck_impute(miss_df, year_cols, random_state=random_state)
    y_pred = hotdeck_df.to_numpy()[eval_mask]
    rows.append({"Method": "Hot-Deck", **_metrics(y_true, y_pred)})
    imputed_results["HotDeck"] = hotdeck_df

    # Linear Interpolation
    lin_df = linear_interp_impute(miss_df, year_cols)
    y_pred = lin_df.to_numpy()[eval_mask]
    rows.append({"Method": "Linear Interpolation", **_metrics(y_true, y_pred)})
    imputed_results["LinearInterpolation"] = lin_df

    # Multiple Imputation
    mi_avg_df, mi_draws = multiple_impute_iterative(
        miss_df, year_cols, n_imputations=n_imputations, random_state=random_state
    )
    y_pred = mi_avg_df.to_numpy()[eval_mask]
    rows.append({"Method": f"Multiple Imputation (avg {n_imputations})", **_metrics(y_true, y_pred)})
    imputed_results["MultipleImputation_avg"] = mi_avg_df
    imputed_results.update(mi_draws)

    # KNN Imputation
    knn_df = knn_impute(miss_df, year_cols, n_neighbors=knn_neighbors, weights="uniform")
    y_pred = knn_df.to_numpy()[eval_mask]
    rows.append({"Method": f"KNN (k={knn_neighbors})", **_metrics(y_true, y_pred)})
    imputed_results["KNN"] = knn_df

    # LSTM Imputation (optional if TF available)
    try:
        lstm_df = lstm_impute(
            miss_df, year_cols,
            random_state=random_state,
            epochs=lstm_epochs,
            batch_size=lstm_batch,
            hidden_units=lstm_hidden,
            verbose=lstm_verbose
        )
        y_pred = lstm_df.to_numpy()[eval_mask]
        rows.append({"Method": f"LSTM AE (epochs={lstm_epochs})", **_metrics(y_true, y_pred)})
        imputed_results["LSTM"] = lstm_df
    except ImportError as e:
        # If TF not installed, skip gracefully
        print("LSTM skipped:", e)

    metrics_df = pd.DataFrame(rows).set_index("Method")

    # Save if asked, otherwise skip silently
    save_base = Path(save_dir) if save_dir is not None else None
    _maybe_save(metrics_df, save_base / "ado_imputation_metrics_ycols.csv" if save_base else None)
    for key, name in [
        ("Mean", "ado_imputed_mean.csv"),
        ("Median", "ado_imputed_median.csv"),
        ("Mode", "ado_imputed_mode.csv"),
        ("HotDeck", "ado_imputed_hotdeck.csv"),
        ("LinearInterpolation", "ado_imputed_linear.csv"),
        ("MultipleImputation_avg", "ado_imputed_MI_avg.csv"),
        ("KNN", "ado_imputed_knn.csv"),
        ("LSTM", "ado_imputed_lstm.csv"),
    ]:
        if key in imputed_results:
            _maybe_save(imputed_results[key], save_base / name if save_base else None)

    print("\n=== Ado_out_school Imputation Metrics (y2001..y2023) ===\n")
    print(metrics_df)

    return metrics_df, imputed_results


In [ ]:
metrics, results = run_imputations_and_evaluate(
    Ado_out_school_Missing,
    Ado_out_school,
    year_start=2001,
    year_end=2023,
    random_state=42,
    n_imputations=5,
    save_dir=".",          # or None to skip saving
    knn_neighbors=5,
    lstm_epochs=150,       # tune if you like punishment
    lstm_hidden=32,
    lstm_batch=64,
    lstm_verbose=0
)
